# Module 01 - Azure ML foundations

Connect to the workspace and register the training data as a versioned asset.
See [README.md](README.md) for the concepts.

In [ ]:
# Tutorial setup: find the repo root and make the repo code importable.
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "src").exists():
            return p
    return start


REPO = find_repo_root(Path.cwd())
for sub in ["src", "data", "webapp/backend"]:
    sys.path.insert(0, str(REPO / sub))
print("Repo root:", REPO)

## Connect with MLClient

`get_ml_client()` reads `.azure-resources.json` and authenticates with
`DefaultAzureCredential` (your `az login`).

In [ ]:
from common.workspace import get_ml_client, load_resources

cfg = load_resources()
ml_client = get_ml_client()
print("Workspace:", ml_client.workspace_name)
print("Resource group:", ml_client.resource_group_name)

## Register the versioned data asset

Registering the CSV as a `uri_file` asset gives every future training run a
precise data version to record.

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

name = cfg.get("data_asset_name", "aeso-hourly-prices")
asset = ml_client.data.create_or_update(
    Data(
        name=name,
        type=AssetTypes.URI_FILE,
        path=str(REPO / "data" / "aeso_hourly.csv"),
        description="Hourly AESO pool price + drivers for day-ahead forecasting.",
        tags={"market": "AESO", "granularity": "hourly"},
    )
)
print(f"Registered {asset.name}:{asset.version}")

## Inspect versions and lineage

Each version is an immutable snapshot you can trace a model back to.

In [ ]:
for d in ml_client.data.list(name=name):
    print(d.name, "latest version:", d.latest_version)

## Next

Continue to [Module 02](../02-training-and-tracking/README.md).